# IMPERANDI × TCGA-LIHC — Light demo

A quick public walkthrough of IMPERANDI using three TCGA-LIHC patients. This version installs the **base package only** and demonstrates metadata ingestion and contrast-phase curation without image conversion, segmentation, or model downloads.

For the complete image-processing workflow, open `demo_full.ipynb`.

## 1. Install IMPERANDI

The IDC downloader is used only to fetch the public TCGA-LIHC DICOM subset.

In [1]:
%cd /content
![ -d /content/IMPERANDI/.git ] || git clone --depth 1 --branch main https://github.com/dmandache/IMPERANDI.git
%cd /content/IMPERANDI
!python -m pip install .
!python -m pip install idc-index


/content
Cloning into 'IMPERANDI'...
remote: Enumerating objects: 197, done.
remote: Counting objects: 100% (197/197), done.
remote: Compressing objects: 100% (179/179), done.
remote: Total 197 (delta 9), reused 77 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (197/197), 3.04 MiB | 8.78 MiB/s, done.
Resolving deltas: 100% (9/9), done.
/content/IMPERANDI
Processing /content/IMPERANDI
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 79.2 MB/s eta 0:00:00
  Created wheel for imperandi: filename=imperandi-0.0.1.dev1+g5587ea092-py3-none-any.whl size=190944 sha256=a88a1019cc0fcc92efcc8f2e448068de7cc7c2a5ce6d

## 2. Download the same three public patients used by both demos

`TCGA-BC-A10Y`, `TCGA-DD-A113`, and `TCGA-DD-A4NJ` are deliberately fixed so results can be compared between the light and full notebooks.

In [2]:
!python examples/benchmarks/download.py /content/TCGA-LIHC-demo \
  --patient TCGA-BC-A10Y \
  --patient TCGA-DD-A113 \
  --patient TCGA-DD-A4NJ

{
  "idc_version": "v25",
  "idc_index_version": "0.13.0",
  "patients": 3,
  "series": 92,
  "estimated_size_MB": 2754.683624,
  "dry_run": false
}
2026-10-08 12:51:42,007 - Disk size needed: 2.75 GB
2026-10-08 12:51:42,007 - Disk size available: 93.22 GB
2026-10-08 12:52:10,112 - 
stoud from s5cmd sync dry run is not empty. Parsing the output to
evaluate what to download and corresponding size with only series level precision

2026-10-08 12:52:10,112 - Parsing the s5cmd sync dry run output...
2026-10-08 12:52:10,197 - Parsing the s5cmd sync dry run output finished
2026-10-08 12:52:10,197 - sync_size (MB): 2754.68
2026-10-08 12:52:10,202 - Initial size of the directory: 0 bytes
2026-10-08 12:52:10,202 - Approximate size of the files that need to be downloaded: 2.75 GB


### Inspect the downloaded series

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

selection = pd.read_csv('/content/TCGA-LIHC-demo/idc_selection.csv')
display(selection[['PatientID','Modality','StudyInstanceUID','SeriesInstanceUID','series_size_MB']])
display(
    selection.groupby(['PatientID', 'Modality'])
    .agg(
        studies=('StudyInstanceUID', 'nunique'),
        series=('SeriesInstanceUID', 'nunique'),
    )
)

,PatientID,Modality,StudyInstanceUID,SeriesInstanceUID,series_size_MB
0,TCGA-BC-A10Y,MR,1.3.6.1.4.1.14519.5.2.1.8421.4008.228155649416...,1.3.6.1.4.1.14519.5.2.1.8421.4008.112954049272...,7.527276
1,TCGA-BC-A10Y,MR,1.3.6.1.4.1.14519.5.2.1.8421.4008.228155649416...,1.3.6.1.4.1.14519.5.2.1.8421.4008.146521872880...,4.557298
2,TCGA-BC-A10Y,MR,1.3.6.1.4.1.14519.5.2.1.8421.4008.228155649416...,1.3.6.1.4.1.14519.5.2.1.8421.4008.210105385864...,3.349438
3,TCGA-BC-A10Y,MR,1.3.6.1.4.1.14519.5.2.1.8421.4008.228155649416...,1.3.6.1.4.1.14519.5.2.1.8421.4008.222111668590...,4.557388
4,TCGA-BC-A10Y,MR,1.3.6.1.4.1.14519.5.2.1.8421.4008.228155649416...,1.3.6.1.4.1.14519.5.2.1.8421.4008.235620519037...,7.526258
...,...,...,...,...,...
87,TCGA-DD-A4NJ,MR,1.3.6.1.4.1.14519.5.2.1.3344.4008.368700536079...,1.3.6.1.4.1.14519.5.2.1.3344.4008.312823803664...,57.948674
88,TCGA-DD-A4NJ,MR,1.3.6.1.4.1.14519.5.2.1.3344.4008.368700536079...,1.3.6.1.4.1.14519.5.2.1.3344.4008.315096169381...,6.418184
89,TCGA-DD-A4NJ,MR,1.3.6.1.4.1.14519.5.2.1.3344.4008.368700536079...,1.3.6.1.4.1.14519.5.2.1.3344.4008.345436872573...,34.862056
90,TCGA-DD-A4NJ,MR,1.3.6.1.4.1.14519.5.2.1.3344.4008.368700536079...,1.3.6.1.4.1.14519.5.2.1.3344.4008.377152448922...,4.420734


studies  series
PatientID    Modality                 
TCGA-BC-A10Y MR              2      24
TCGA-DD-A113 CT              4      17
             MR              1      10
TCGA-DD-A4NJ CT              3      22
             MR              1      19

## 3. Ingest: DICOM → harmonized volume-level cohort

`imperandi ingest` parses the DICOM collection, reconstructs volume-level records, and runs the modality-specific metadata engines. The raw metadata evidence (for example `rule_phase`) is retained so the next task can resolve the canonical phase explicitly.

In [4]:
!mkdir -p /content/imperandi-demo/tables
!imperandi ingest \
  /content/TCGA-LIHC-demo \
  /content/imperandi-demo/tables \
  --manifest examples/demos/demo_manifest.yaml \
  --id_source tags \
  --patient_key_from PatientID \
  --study_id_from StudyInstanceUID \
  --series_id_from SeriesInstanceUID \
  --num_workers 2

2026-10-08 12:53:05 | INFO     | imperandi.cli | Running parse.py with arguments:
{   'archive_cache_dir': None,
    'archive_detect_sample_size': 128,
    'archive_max_depth': 3,
    'checkpoint_every_rows': 10000,
    'checkpoint_every_sec': 300,
    'command': 'ingest',
    'csv_path_out': None,
    'dry_run': False,
    'force_dicom_read': False,
    'id_source': 'tags',
    'keep_archive_cache': False,
    'log_file': None,
    'log_level': None,
    'manifest': 'examples/demos/demo_manifest.yaml',
    'num_workers': 2,
    'output_dir': '/content/imperandi-demo/tables',
    'patient_key_from': 'PatientID',
    'quiet': False,
    'resume': True,
    'root_path': '/content/TCGA-LIHC-demo',
    'series_id_from': 'SeriesInstanceUID',
    'snapshot_sample_size': 500,
    'snapshot_seed': 42,
    'snapshot_tags': False,
    'strict_resume': False,
    'study_id_from': 'StudyInstanceUID',
    'tags': '',
    'verbose': False}
2026-10-08 12:53:05 | INFO     | imperandi.cli | Running cle

### Inspect ingest outputs

In [5]:
raw = pd.read_csv('/content/imperandi-demo/tables/dicom_index.csv')
clean = pd.read_csv('/content/imperandi-demo/tables/dicom_index_clean.csv')

print(f'Parsed DICOM rows: {len(raw):,}')
print(f'Curated volume rows: {len(clean):,}')

print('Volumes by patient and modality:')
display(
    clean.groupby(['patient_key', 'Modality'])
    .agg(
        studies=('study_id', 'nunique'),
        series=('series_id', 'nunique'),
        volumes=('volume_id', 'nunique'),
    )
)

print('Representative curated volumes:')
clean_cols = [c for c in ['patient_key', 'date', 'Modality', 'SeriesDescription', 'volume_id'] if c in clean.columns]
display(clean[clean_cols].head(10))



Parsed DICOM rows: 5,931
Curated volume rows: 66
Volumes by patient and modality:


studies  series  volumes
patient_key  Modality                          
TCGA-BC-A10Y MR              2       4        4
TCGA-DD-A113 CT              4      13       14
             MR              1       8       13
TCGA-DD-A4NJ CT              3      15       15
             MR              1      15       20

Representative curated volumes:


,patient_key,date,Modality,SeriesDescription,volume_id
0,TCGA-BC-A10Y,1993-03-21,MR,VIBE 3D NEW 2002B,8e645fafd837e284d640f444a606d53608916369
1,TCGA-BC-A10Y,1993-03-21,MR,Subtraction_S11_S9_1,30d17d0c04eaf4f2862f554af44e42fa50e138af
2,TCGA-BC-A10Y,1993-03-21,MR,VIBE 3D NEW 2002B,74e6d13efd2a9cc8e61ffb1bb3e97938800a273a
3,TCGA-BC-A10Y,1993-05-01,MR,VIBE 3D NEW 2002B,3f3f02cda686c10767919269bc39d3396d4162f3
4,TCGA-DD-A113,1998-12-09,MR,ax FSE,7b572adfb1df6d1103fb7a9c0c17b4499552ff8c
5,TCGA-DD-A113,1998-12-09,MR,cor ssfse,ca27c5ba3b7ccbfe98a50db66871d82b40ab8a96
6,TCGA-DD-A113,1998-12-09,MR,improved IP/OP,8fc5aa6d8e71c0aa2dd10f69ff5bbe4204be4050
7,TCGA-DD-A113,1998-12-09,MR,improved IP/OP,dee909af88c497d475069cb88923edabf5fd24cf
8,TCGA-DD-A113,1998-12-09,MR,lava pre,286fed735c02466d126c548eb9e7b8f9debc2b6f
9,TCGA-DD-A113,1998-12-09,MR,realtime,0c05b6b4c6be2afbb405ecbdc415fd6a13e25abf


## 4. Resolve canonical contrast phases from metadata

The demo uses IMPERANDI's metadata rules only: no image conversion, TotalSegmentator, or model weights are required. The phase task consumes the metadata evidence generated during ingest and writes canonical `phase`, provenance, and best-candidate outputs.

In [6]:
!imperandi phase \
  /content/imperandi-demo/tables/dicom_index_clean.csv \
  --csv_path_out /content/imperandi-demo/tables/dicom_index_phased.csv \
  --manifest examples/demos/demo_manifest.yaml

2026-10-08 12:53:30 | INFO     | imperandi.cli | Running phase.py with arguments:
{   'checkpoint_every_rows': 50,
    'checkpoint_every_sec': 300,
    'command': 'phase',
    'csv_path': '/content/imperandi-demo/tables/dicom_index_clean.csv',
    'csv_path_out': '/content/imperandi-demo/tables/dicom_index_phased.csv',
    'dry_run': False,
    'error_csv_path': '/content/imperandi-demo/tables/phase_errors.csv',
    'force': False,
    'log_file': None,
    'log_level': None,
    'manifest': 'examples/demos/demo_manifest.yaml',
    'quiet': False,
    'resume': True,
    'selected_csv_path': None,
    'strict_resume': False,
    'verbose': False}
2026-10-08 12:53:31 | INFO     | imperandi.extract.phase | Phase strategy order: metadata_phase_candidates (rules)
Phase: 0it [00:00, ?it/s]
2026-10-08 12:53:31 | INFO     | imperandi.extract.phase | Phase strategy 1/1: metadata_phase_candidates (rules) resolved 22 volume(s); 19 unresolved
2026-10-08 12:53:31 | INFO     | imperandi.extract.pha

### Inspect phase labels and best-per-exam selection

In [ ]:
phased = pd.read_csv('/content/imperandi-demo/tables/dicom_index_phased.csv')
selected = pd.read_csv('/content/imperandi-demo/tables/dicom_index_clean_curated.csv')

phase_counts = (
    phased.groupby(['Modality', 'phase'], dropna=False)
    .size()
    .rename('volumes')
    .reset_index()
)
print('Phase distribution by modality:')
display(phase_counts)

phase_plot = phase_counts.pivot(index='phase', columns='Modality', values='volumes').fillna(0)
ax = phase_plot.plot.bar()
ax.set_xlabel('Phase')
ax.set_ylabel('Volumes')
ax.set_title('Curated phase distribution')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

if 'phase_source' in phased.columns:
    print('Phase provenance:')
    display(phased['phase_source'].value_counts(dropna=False).rename('volumes').to_frame())

if 'phase_status' in phased.columns:
    print('Phase status:')
    display(phased['phase_status'].value_counts(dropna=False).rename('volumes').to_frame())

    unresolved = phased.loc[phased['phase_status'].eq('UNRESOLVED')].copy()

    unresolved_ct = unresolved.loc[unresolved['Modality'].eq('CT')]
    ct_cols = [
        c for c in [
            'patient_key', 'date', 'volume_id', 'SeriesDescription',
            'phase_text_column', 'phase_text_value', 'phase_source', 'phase_reason',
            'rule_phase_reason',
            'ct_phase', 'ct_phase_reason', 'ct_phase_confidence',
            'selection_score', 'phase', 'phase_status'
        ]
        if c in unresolved_ct.columns
    ]
    print(f'Unresolved CT curation examples ({len(unresolved_ct)} total):')
    display(unresolved_ct[ct_cols].head(10))

    unresolved_mr = unresolved.loc[unresolved['Modality'].isin(['MR', 'MRI'])]
    mr_cols = [
        c for c in [
            'patient_key', 'date', 'volume_id', 'SeriesDescription',
            'phase_text_column', 'phase_text_value', 'phase_source', 'phase_reason',
            'rule_phase_reason',
            'mri_perfusion_source', 'mri_dynamic_order_source',
            'mri_dynamic_timing_source', 'mri_dynamic_timing_reliable',
            'mri_sequence', 'plane',
            'mri_perfusion_label', 'mri_perfusion_reason',
            'mri_perfusion_confidence',
            't1_score', 't2_score', 'dwi_score', 'selection_score',
            'phase', 'phase_status'
        ]
        if c in unresolved_mr.columns
    ]
    print(f'Unresolved MR curation examples ({len(unresolved_mr)} total):')
    display(unresolved_mr[mr_cols].head(10))

if 'phase_source' in phased.columns:
    resolved = (phased['phase_status'].eq('RESOLVED') & phased['phase_source'].eq('metadata_phase_candidates')).sum()
    print(f'Metadata-resolved phases: {resolved}/{len(phased)}')
    assert resolved > 0, 'No phases were resolved from metadata; inspect qc_unresolved_phase.csv.'

selected['selection_score'] = pd.to_numeric(selected['selection_score'], errors='coerce')
scores = selected['selection_score'].dropna()
print(f'Best-per-exam curated rows: {len(selected)}')

curated_cols = [
    c for c in [
        'patient_key', 'date', 'Modality', 'SeriesDescription',
        'mri_perfusion_source', 'mri_dynamic_order_source',
            'mri_dynamic_timing_source', 'mri_dynamic_timing_reliable',
            'mri_sequence', 'phase', 'selection_score'
    ]
    if c in selected.columns
]
ranked = selected.dropna(subset=['selection_score'])

print('Highest-scoring curated selections:')
display(ranked.nlargest(10, 'selection_score')[curated_cols])

print('Lowest-scoring curated selections:')
display(ranked.nsmallest(10, 'selection_score')[curated_cols])


## What this demonstrated

The light workflow stops here: public heterogeneous DICOMs → standardized identifiers → reconstructed volumes → phase/sequence annotation → reviewable best-candidate selection. The full notebook continues with NIfTI conversion, segmentation, and interactive visual QC.